# Ingerindo os dados brutos da landing para a Bronze 🥉
- O objetivo dessa etapa é realizar a leitura dos arquivos brutos armazenados na `landing` (csv's, excel, txt, etc) e armazenar os dados "crus" na `bronze`, só que de forma padronizada, usando o formato delta
- É aqui que começa a **criação das tabelas** que darão **origem** a camada `silver` e `gold`
- Foi adicionado nessa etapa o conceito de **ingestão incremental** 
  - A ingestão incremental trata-se do fato de adicionar dados novos ao que já existe 
  - É uma prática comum usar o overwrite na escrita, entretanto, o que isso faz é sobrescever completamente os dados que já existiam pelos novos dados que já estão chegando 
  - Entretanto, ao usar a ideia do append, que vem com a incrementalidade, o que acontece é que os dados antigos permanecerão lá enquanto apenas os dados novos serão adicioandos 

In [0]:
# Variáveis globais que serão utilizadas na criação de população das tabelas da bronze 
catalog = "medalhao_at"
schema = "bronze"

## 1️⃣ Realizando o drop das tabelas antigas

In [0]:
# Função responsável pra realizar o drop das tabelas da bronze
def drop_tables(lista: list):
  for table_name in lista:
    drop_query = f"DROP TABLE IF EXISTS {catalog}.{schema}.{table_name}"
    spark.sql(drop_query)

In [0]:
## Obs: Esse bloco de código só deve ser executado em caso de restart completo do pipeline ou em caso de primeira execução

# Nome das tabelas que estão sendo utilizadas
tabelas_drop = [
    "tb_customers", 
    "tb_geolocalizacao", 
    "tb_order_items", 
    "tb_order_payments", 
    "tb_order_reviews", 
    "tb_orders", 
    "tb_products", 
    "tb_sellers", 
    "tb_product_category_name_translation",
    "tb_cotacao_dolar"
]

# Chamada da função para dropar todas as tabelas
drop_tables(tabelas_drop)

## 2️⃣ Realizando leitura dos dados da landing 

In [0]:
# Função responsável para realizar a leitura dos dados da landing
def ler_landing(filepath):
  path = f"/Volumes/medalhao_at/default/landing/{filepath}"
  df = spark.read.format("csv") \
  .option("header", "true") \
  .option("inferSchema", "true") \
  .option("multiLine", "true") \
  .option("escape", '"') \
  .load(path)

  return df

In [0]:
# O bloco de código abaixo serve como verificação de dados na landing
# O objetivo dele é verificar se os dados estão lá mostrando uma pequena quantidade de dados

# Essa variável pode ser mudada a vontade, inclusive pode não existir se for mais conveniente ao usuário
qtd_linhas = 10

csvs = [
    "olist_customers_dataset.csv", 
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", 
    "olist_order_payments_dataset.csv", 
    "olist_order_reviews_dataset.csv", 
    "olist_orders_dataset.csv", 
    "olist_products_dataset.csv", 
    "olist_sellers_dataset.csv", 
    "product_category_name_translation.csv"
]

for csv in csvs:
    df = ler_landing(csv)
    print(f"\nCSV: {csv}\n")
    display(df.limit(qtd_linhas))


CSV: olist_customers_dataset.csv



customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_name,customer_gender,customer_birth_date,customer_age
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Zoe Nogueira,F,1956-09-07,70
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Liam Viana,M,1974-09-23,52
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Diego Aparecida,M,1964-05-20,62
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Marcos Vinicius Azevedo,M,1967-01-14,59
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Srta. Juliana Siqueira,F,1950-08-01,76
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,Dra. Alice Lopes,F,1975-07-07,51
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP,Maysa Vieira,F,1998-09-27,28
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,José Miguel Freitas,M,2004-01-08,22
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,Samuel Duarte,M,1991-08-22,35
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,Rael Pimenta,M,1995-11-12,31



CSV: olist_geolocation_dataset.csv



geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP
1041,-23.5443921648681,-46.63949930627844,sao paulo,SP
1035,-23.541577961711493,-46.64160722329613,sao paulo,SP
1012,-23.547762303364266,-46.63536053788448,são paulo,SP
1047,-23.546273112412678,-46.64122516971552,sao paulo,SP
1013,-23.546923208436723,-46.6342636964915,sao paulo,SP
1029,-23.543769055769133,-46.63427784085132,sao paulo,SP
1011,-23.547639550320632,-46.63603162315495,sao paulo,SP



CSV: olist_order_items_dataset.csv



...

product_id,product_name,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,Perfume Premium,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
3aa071139cb16b67ca9e5dea641aaa2f,Conjunto de Pincéis,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
96bd76ec8810374ed1b65e291975717f,Barraca de Camping,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
cef67bcfe19066a932b7673e239eb23d,Chupeta Premium,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
9dc1a7de274444849c219cff195d0b71,Vassoura Mágica,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
41d3672d4792049fa1779bb35283ed13,Violão Acústico,instrumentos_musicais,60.0,745.0,1.0,200.0,38.0,5.0,11.0
732bd381ad09e530fe0a5f457d81becb,Almofada Decorativa Azul,cool_stuff,56.0,1272.0,4.0,18350.0,70.0,24.0,44.0
2548af3e6e77a690cf3eb6368e9ab61e,Tapete Felpudo Plus,moveis_decoracao,56.0,184.0,2.0,900.0,40.0,8.0,40.0
37cc742be07708b53a98702e77a21a02,Máquina de Lavar Ultra,eletrodomesticos,57.0,163.0,1.0,400.0,27.0,13.0,17.0
8c92109888e8cdf9d66dc7e463025574,Boneca Ultra,brinquedos,36.0,1156.0,1.0,600.0,17.0,10.0,12.0



CSV: olist_sellers_dataset.csv



seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_name,seller_registration_date
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,Dr. Pedro Miguel Vasconcelos,2018-06-25
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,Gabriela da Cruz,2018-09-21
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,Luigi Viana,2018-04-03
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,Beatriz Brito,2018-12-20
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,Gabrielly Pastor,2017-08-24
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ,Rebeca Lopes,2018-02-22
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE,Lorenzo Borges,2017-06-22
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP,Letícia Abreu,2018-05-22
768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP,Nicole Barbosa,2017-10-24
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR,Sr. Henry Gabriel Pinto,2017-12-31



CSV: product_category_name_translation.csv



product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


## 3️⃣ Realizando ingestão dos dados na camada bronze 

In [0]:
from pyspark.sql import functions as F

# Função responsável por realizar a ingestão dos dados da landing para a camada bronze
def ingest_csv(filename, filepath):
  try :
    df = ler_landing(filepath)

    if df.count() == 0:
      raise ValueError(f"O arquivo {filepath} está vazio ou não pôde ser lido")

    df_with_metadata = df.withColumn("timestamp_ingestion", F.from_utc_timestamp(F.current_timestamp(), "America/Recife"))

    df_with_metadata.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.{filename}")

    print(f"✅ A tabela bronze.{filename} foi criada com sucesso!!")
  except Exception as e :
    print(f"❌ Ocorreu um erro durante a ingestão do arquivo {filepath}: {e}")


In [0]:
tabelas_ingestao = [
    "tb_customers", 
    "tb_geolocalizacao", 
    "tb_order_items", 
    "tb_order_payments", 
    "tb_order_reviews", 
    "tb_orders", 
    "tb_products", 
    "tb_sellers", 
    "tb_product_category_name_translation"
]

csvs = [
    "olist_customers_dataset.csv", 
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv", 
    "olist_order_payments_dataset.csv", 
    "olist_order_reviews_dataset.csv", 
    "olist_orders_dataset.csv", 
    "olist_products_dataset.csv", 
    "olist_sellers_dataset.csv", 
    "product_category_name_translation.csv"
]

for tabela, csv in zip(tabelas_ingestao, csvs):
    ingest_csv(tabela, csv)

print("🚀 Tabelas da camada bronze ingeridas com sucesso !!")

✅ A tabela bronze.tb_customers foi criada com sucesso!!
✅ A tabela bronze.tb_geolocalizacao foi criada com sucesso!!
✅ A tabela bronze.tb_order_items foi criada com sucesso!!
✅ A tabela bronze.tb_order_payments foi criada com sucesso!!
✅ A tabela bronze.tb_order_reviews foi criada com sucesso!!
✅ A tabela bronze.tb_orders foi criada com sucesso!!
✅ A tabela bronze.tb_products foi criada com sucesso!!
✅ A tabela bronze.tb_sellers foi criada com sucesso!!
✅ A tabela bronze.tb_product_category_name_translation foi criada com sucesso!!
🚀 Tabelas da camada bronze ingeridas com sucesso !!


## 4️⃣ Ingestão via API

In [0]:
from pyspark.sql import functions as F

# Determinando os valores a serem usados pra buscar a cotação do dólar

# Após análise manual sobre cada csv individual, percebeu-se que o melhor csv pra definir essa métrica era o `olist_orders_dataset.csv`

# Após isso foi feita uma análise 
# 1. Caminho do arquivo (Ajuste para o local onde você salvou o CSV na sua conta)
csv_escolhido = "/Volumes/medalhao_at/default/landing/olist_orders_dataset.csv" 

# 2. Lendo o dataset de pedidos
df_orders = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .load(csv_escolhido)

# 3. Descobrindo a data mínima e máxima de compra e formatando para MM-DD-YYYY
datas = df_orders.select(
    F.date_format(F.min("order_purchase_timestamp"), "MM-dd-yyyy").alias("data_inicio"),
    F.date_format(F.max("order_purchase_timestamp"), "MM-dd-yyyy").alias("data_fim")
).collect()[0]

# 4. Salvando os resultados em variáveis Python
data_analise_inicio = datas["data_inicio"]
data_analise_fim = datas["data_fim"]

# 5. Exibindo os resultados para você conferir
print("=== Período de Vendas da Olist ===")
print(f"Data Inicial encontrada: {data_analise_inicio}")
print(f"Data Final encontrada: {data_analise_fim}")
print("==================================")

# Agora você já pode passar essas variáveis direto para a sua função:
# ingest_api_dolar(data_inicio, data_fim, "api_cotacao_dolar", catalog_name, schema_name)

=== Período de Vendas da Olist ===
Data Inicial encontrada: 09-04-2016
Data Final encontrada: 10-17-2018


In [0]:
# As datas abaixo foram escolhidas após analisar os dados de cada um dos csvs enviados 

# Após essa análise, concluiu-se que dentre os dados enviados, aquele que apresentava informação mais relevante para construção da tabela `tb_cotacao_dolar` era o conjunto olist_orders_dataset.csv

# A escolha foi feita ao perceber que ele contém o histórico de todos os pedidos. Além disso, a coluna `order_purchase_timestamp` (data da compra) é quem dita quando o dinheiro mudou de mãos e, portanto, quando a conversão da moeda importa

dbutils.widgets.text("data_inicio", data_analise_inicio)
dbutils.widgets.text("data_fim", data_analise_fim)

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8190142110349827>, line 7
      1 # As datas abaixo foram escolhidas após analisar os dados de cada um dos csvs enviados 
      2 
      3 # Após essa análise, concluiu-se que dentre os dados enviados, aquele que apresentava informação mais relevante para construção da tabela `tb_cotacao_dolar` era o conjunto olist_orders_dataset.csv
      4 
      5 # A escolha foi feita ao perceber que ele contém o histórico de todos os pedidos. Além disso, a coluna `order_purchase_timestamp` (data da compra) é quem dita quando o dinheiro mudou de mãos e, portanto, quando a conversão da moeda importa
----> 7 dbutils.widgets.text("data_inicio", data_analise_inicio)
      8 dbutils.widgets.text("data_fim", data_analise_fim)
     10 data_inicio = dbutils.widgets.get("data_inicio")

NameError: name 'data_analise_inicio' is not defined

In [0]:
import requests 
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# Função dedicada a buscar a cotação do dólar em determinado período de tempo especificado
def get_cotacao_dolar(data_inicio_formatada, data_fim_formatada):
    try :
        url = (
            f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"
        )
        response = requests.get(url)
        if response.status_code != 200:
            raise Exception(f"❌ Erro ao fazer a requisição: {response.status_code}")
        data = response.json().get("value", [])
        return data
    except Exception as e:
        print(f"❌ Ocorreu um erro durante a ingestão do arquivo: {e}")

# Função dedicada a realizar a ingestão dos dados buscados pela função get_cotacao_dolar
def ingest_api_dolar(data_inicio, data_fim, table_name, incremental="nao"):
    try:
        print(f"Buscando cotações do dólar entre {data_inicio} e {data_fim}...")
        
        # 1. Chama a função que você criou para pegar os dados
        dados_json = get_cotacao_dolar(data_inicio_formatada=data_inicio,data_fim_formatada=data_fim)
        
        if not dados_json:
            raise ValueError("A API não retornou dados para esse período ou a lista está vazia.")

        # 2. Define o schema esperado (Garante que o PySpark entenda os tipos de dados)
        schema_cotacao = StructType([
            StructField("dataHoraCotacao", StringType(), True),
            StructField("cotacaoCompra", DoubleType(), True)
        ])

        # 3. Transforma a lista de dicionários Python em um DataFrame do Spark
        df = spark.createDataFrame(dados_json, schema=schema_cotacao)

        # 3.5. Converte a coluna de String para Timestamp de verdade
        df = df.withColumn(
            "dataHoraCotacao", 
            F.to_timestamp(F.col("dataHoraCotacao"), "yyyy-MM-dd HH:mm:ss.SSS")
        )

        # 4. Adiciona a coluna de metadados
        df_with_metadata = df.withColumn(
            "timestamp_ingestion", 
            F.from_utc_timestamp(F.current_timestamp(), "America/Recife")
        )

        # 5. Salva na camada Bronze usando sua regra de overwrite/append
        table_path = f"{catalog}.{schema}.{table_name}"
        
        if incremental == "nao": 
            df_with_metadata.write.format("delta").mode("overwrite").saveAsTable(table_path)
        elif incremental == "sim":
            df_with_metadata.write.format("delta").mode("append").saveAsTable(table_path)
        else:
            raise ValueError("O tipo do incremental não foi definido corretamente.")

        print(f"✅ A tabela {table_path} foi criada com sucesso!!")
        
        # Exibe uma amostra dos dados salvos
        display(df_with_metadata.limit(10))

    except Exception as e:
        print(f"❌ Ocorreu um erro durante a ingestão da API: {e}")


ingest_api_dolar(
    data_inicio=data_inicio, 
    data_fim=data_fim, 
    table_name="tb_cotacao_dolar", 
    incremental="nao"
)

Buscando cotações do dólar entre 09-04-2016 e 10-17-2018...
✅ A tabela medalhao_at.bronze.tb_cotacao_dolar foi criada com sucesso!!


dataHoraCotacao,cotacaoCompra,timestamp_ingestion
2016-09-05T13:09:55.659Z,3.2715,2026-04-04T08:37:53.124Z
2016-09-06T13:02:39.984Z,3.2446,2026-04-04T08:37:53.124Z
2016-09-08T13:03:53.968Z,3.1928,2026-04-04T08:37:53.124Z
2016-09-09T13:14:00.885Z,3.2632,2026-04-04T08:37:53.124Z
2016-09-12T13:08:01.541Z,3.2848,2026-04-04T08:37:53.124Z
2016-09-13T13:03:56.534Z,3.2966,2026-04-04T08:37:53.124Z
2016-09-14T13:05:51.819Z,3.3256,2026-04-04T08:37:53.124Z
2016-09-15T13:08:34.825Z,3.332,2026-04-04T08:37:53.124Z
2016-09-16T13:04:11.365Z,3.2998,2026-04-04T08:37:53.124Z
2016-09-19T13:07:42.039Z,3.263,2026-04-04T08:37:53.124Z
